# Semana 9: Apuntes de la clase

**Modelos de valoración de activos: CAPM, SML y multifactoriales** (CFA L2, Portfolio Management: *Using Multifactor Models*; CFA L1: *Portfolio Risk and Return: Part II* como punto de partida)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/09_capm_multifactor/clase09_apuntes.ipynb)

Este notebook resume los conceptos que debiste llevarte de la clase, con los ejemplos numéricos ejecutables. Cierra el círculo del CAPM (lo usamos en la semana 2, lo justificamos en las semanas 6 y 7) y abre el de los modelos de factores, que es el lenguaje con que hoy se describe cualquier fondo. Entra en la PC2 (miércoles 11/11).

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas y símbolos que usamos esta semana; los de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| CAPM | *Capital Asset Pricing Model* | En equilibrio, el retorno esperado de un activo es lineal en su beta: $E(R_i) = R_f + \beta_i[E(R_m) - R_f]$. |
| CML | *Capital Market Line* (línea del mercado de capitales) | La CAL cuando el portafolio riesgoso es el mercado. Eje $\sigma$; solo portafolios eficientes. |
| SML | *Security Market Line* (línea del mercado de valores) | La gráfica del CAPM. Eje $\beta$; vale para todos los activos y portafolios. |
| $\beta$ | Beta | $\sigma_{im}/\sigma_m^2 = \rho_{im}\sigma_i/\sigma_m$: cuánto riesgo sistemático carga el activo. |
| $\alpha$ | Alfa | Retorno pronosticado (o realizado) menos el exigido por el modelo. Sobre la SML, alfa positivo: subvaluado. |
| $R^2$ | Coeficiente de determinación | En la regresión contra el mercado, la fracción de la varianza que es sistemática. |
| APT | *Arbitrage Pricing Theory* (teoría de valoración por arbitraje) | Retorno esperado lineal en varias sensibilidades a factores; se sostiene por ausencia de arbitraje. |
| $\lambda_k$ | Prima por riesgo del factor $k$ | Lo que paga el mercado por cada unidad de sensibilidad a ese factor. |
| FF3 | Modelo de tres factores de Fama y French | Mercado, SMB y HML. |
| SMB | *Small Minus Big* | Retorno de las empresas pequeñas menos el de las grandes: el factor tamaño. |
| HML | *High Minus Low* | Retorno de las empresas de valor (alto valor libro sobre precio) menos el de las de crecimiento: el factor valor. |
| WML o MOM | *Winners Minus Losers* (*momentum*) | Ganadoras recientes menos perdedoras: el cuarto factor, de Carhart. |
| RMW y CMA | *Robust Minus Weak*, *Conservative Minus Aggressive* | Rentabilidad e inversión: los factores que completan el modelo de cinco. |
| PCA | *Principal Components Analysis* | La técnica detrás de los modelos estadísticos de factores. |
| ETF | *Exchange Traded Fund* | Fondo cotizado; en la práctica usamos varios para leer su estilo. |

## 1. Del tangente al mercado: la CML

Supuestos del CAPM: inversionistas media-varianza a un periodo con las **mismas expectativas**, que prestan y se endeudan a $R_f$, en mercados sin fricciones. Mismas expectativas implican el mismo tangente para todos, y la suma de lo que todos tienen es el mercado: el tangente es el portafolio de mercado $M$. La CAL de la semana 7 con $M$ como portafolio riesgoso es la **CML**:

$$E(R_p) = R_f + \frac{E(R_m) - R_f}{\sigma_m}\,\sigma_p$$

Vale solo para portafolios eficientes (mezclas de $R_f$ y $M$). Usamos $R_f = 4.5\%$, prima de mercado de 5.5% (los de la semana 2), así que $E(R_m) = 10\%$, y $\sigma_m = 16\%$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
from utils.finanzas import capm, alfa_capm, descomponer_riesgo, retorno_multifactor

rf, erp, sigma_m = 4.5, 5.5, 16.0          # todo en %
sharpe_m = erp / sigma_m
print(f"Pendiente de la CML (Sharpe del mercado) = {sharpe_m:.3f}")
print(f"Un portafolio eficiente con sigma = 8%  rinde {rf + sharpe_m * 8:.2f}%  (mitad en M, mitad en rf)")
print(f"Un portafolio eficiente con sigma = 24% rinde {rf + sharpe_m * 24:.2f}% (150% en M, endeudado)")

Salida esperada: pendiente de 0.344; con $\sigma = 8\%$ la CML paga 7.25% y con $\sigma = 24\%$ paga 12.75%.

## 2. El CAPM, la SML y el alfa

Lo que un activo le aporta al riesgo de $M$ es su covarianza con $M$. Estandarizada:

$$\beta_i = \frac{\sigma_{im}}{\sigma_m^2} = \rho_{im}\,\frac{\sigma_i}{\sigma_m} \qquad\qquad E(R_i) = R_f + \beta_i\,[E(R_m) - R_f]$$

La SML vale para **todos** los activos. El retorno **exigido** sale de la SML; el **pronosticado**, del análisis; la diferencia es el **alfa**. Sobre la línea: promete más de lo que su riesgo exige, está subvaluado. Bajo la línea: sobrevalorado.

In [ ]:
activos = pd.DataFrame({"beta": [1.2, 0.6, 1.1], "pronostico": [12.0, 8.0, 10.0], "sigma": [24.0, 12.0, 18.0]},
                       index=["minera", "electrica", "banco"])
activos["exigido (SML)"] = [capm(rf, b, erp) for b in activos["beta"]]
activos["alfa"] = [alfa_capm(p, rf, b, erp) for p, b in zip(activos["pronostico"], activos["beta"])]
activos["lectura"] = np.where(activos["alfa"] > 0, "sobre la SML: subvaluado", "bajo la SML: sobrevalorado")
activos["si fuera eficiente (CML)"] = rf + sharpe_m * activos["sigma"]
print(activos.round(2).to_string())

w_t = np.array([0.244, 0.476, 0.280])        # el tangente de tres activos de la semana 7
print(f"\nBeta del tangente de la semana 7 = {w_t @ activos['beta'].values:.2f} (promedio ponderado de los betas)")

Salida esperada: exigidos de 11.10%, 7.80% y 10.55%; alfas de $+0.90$, $+0.20$ y $-0.55$. El 11.10% de la minera es exactamente el $K_e$ con que descontamos en la unidad 1. Y no confundir las rectas: con $\sigma = 24\%$ la CML pagaría 12.75%; la minera rinde 12: está **bajo** la CML (como todo activo individual, porque parte de su $\sigma$ no se paga) y **sobre** la SML. El beta de un portafolio es el promedio ponderado: 0.89 para el tangente de la semana 7.

## 3. Beta y la descomposición del riesgo

Con el modelo de mercado $R_i = \alpha_i + \beta_i R_m + \varepsilon_i$:

$$\sigma_i^2 = \beta_i^2\sigma_m^2 + \sigma_\varepsilon^2 \qquad\qquad R^2 = \frac{\beta_i^2\sigma_m^2}{\sigma_i^2} = \rho_{im}^2$$

In [ ]:
d = descomponer_riesgo(beta=1.2, sigma_mercado=sigma_m, sigma_activo=24.0)
print(f"Varianza total = {24.0 ** 2:.0f} = sistematica {d['sigma_sistematica'] ** 2:.1f} + idiosincratica {d['sigma_idiosincratica'] ** 2:.1f}")
print(f"sigma sistematica = {d['sigma_sistematica']:.1f}% | sigma idiosincratica = {d['sigma_idiosincratica']:.1f}%")
print(f"R2 = {d['r2']:.2f} | correlacion con el mercado = {d['correlacion']:.2f}")

Salida esperada: $576 = 368.6 + 207.4$; volatilidad sistemática de 19.2% e idiosincrática de 14.4%; $R^2 = 0.64$ y $\rho = 0.80$. En la semana 6 dijimos que de los 24 puntos de la minera solo importaba una parte: son 19.2. Los otros 14.4 se diversifican y nadie los paga. (Ojo: las volatilidades no se suman; las varianzas sí.)

## 4. Lo que el CAPM no explica

La SML empírica es demasiado plana (los betas bajos rinden más de lo predicho y los altos, menos); hay anomalías persistentes (tamaño, valor, *momentum*, rentabilidad, inversión); la crítica de Roll recuerda que el verdadero portafolio de mercado no es observable; y un solo beta mezcla riesgos de naturaleza distinta. La respuesta: más factores.

## 5. APT: la lógica del arbitraje

$$E(R_p) = R_f + \lambda_1\beta_{p,1} + \dots + \lambda_K\beta_{p,K}$$

Tres supuestos: los retornos siguen un modelo de factores, hay muchos activos (el riesgo idiosincrático se diversifica) y no hay arbitraje entre portafolios bien diversificados. No pide expectativas homogéneas ni portafolio de mercado. Tampoco dice cuáles ni cuántos son los factores.

In [ ]:
# Un factor, rf = 4.5%. A y C fijan la prima: lambda = (10 - 4.5) / 1.00 = 5.5
lam = (10.0 - rf) / 1.00
for nombre, b, e in [("A", 1.00, 10.00), ("C", 0.50, 7.25), ("D", 0.75, 9.50)]:
    justo = retorno_multifactor(rf, [b], [lam])
    print(f"{nombre}: sensibilidad {b:.2f} | E(R) = {e:.3f}% | consistente con el factor = {justo:.3f}% | diferencia = {e - justo:+.3f}")

# Arbitraje: largo en D, corto en la mezcla 50/50 de A y C (misma sensibilidad)
b_mezcla, e_mezcla = 0.5 * 1.00 + 0.5 * 0.50, 0.5 * 10.00 + 0.5 * 7.25
print(f"\nMezcla A y C: sensibilidad {b_mezcla:.2f}, E(R) = {e_mezcla:.3f}%")
print(f"Largo D y corto mezcla: inversion neta 0, sensibilidad neta {0.75 - b_mezcla:.2f}, ganancia esperada {9.50 - e_mezcla:+.3f}%")

Salida esperada: D debería rendir 8.625% y ofrece 9.50%. Largo en D y corto en la mezcla de A y C: inversión cero, riesgo de factor cero y ganancia de $+0.875\%$. Al explotarlo el precio de D sube hasta que su retorno esperado cae a 8.625%: el arbitraje es lo que mantiene a todos sobre la línea.

## 6. Tres familias de modelos de factores

| Familia | Qué son los factores | Qué se estima | Ejemplo |
|---|---|---|---|
| Macroeconómicos | **Sorpresas** en variables macro | Las sensibilidades, por regresión | Chen, Roll y Ross |
| Fundamentales | Retornos asociados a **atributos** de las empresas | Las cargas o los retornos de los factores | Fama-French, Carhart, BARRA |
| Estadísticos | Combinaciones de retornos extraídas de los datos | Todo | Componentes principales |

En un modelo macro el intercepto es el **retorno esperado** del activo y lo que mueve al retorno realizado es la sorpresa (dato observado menos dato esperado):

$$R_i = a_i + b_{i,1}F_{PBI} + b_{i,2}F_{INF} + \varepsilon_i$$

In [ ]:
# La minera: retorno esperado 12%, sensibilidad 2.0 al crecimiento y -1.0 a la inflacion
a, b_pbi, b_inf = 12.0, 2.0, -1.0
sorpresa_pbi = 2.0 - 3.0          # crecio 2.0 cuando se esperaba 3.0
sorpresa_inf = 3.0 - 2.5          # inflacion de 3.0 cuando se esperaba 2.5
eps = 1.5                         # evento propio de la empresa

macro = b_pbi * sorpresa_pbi + b_inf * sorpresa_inf
print(f"Retorno realizado = {a} + ({macro:+.1f} de la macro) + ({eps:+.1f} propio) = {a + macro + eps:.1f}%")

Salida esperada: 11.0%. De los $-1.0$ puntos respecto de lo esperado, $-2.5$ vinieron de la macro (sistemático) y $+1.5$ fue propio de la empresa: eso es **atribución de retorno**. Si el PBI hubiera crecido justo lo esperado, ese factor no habría aportado nada.

## 7. Fama-French y el alfa que depende del modelo

$$R_i - R_f = \alpha_i + b_i(R_m - R_f) + s_i\,SMB + h_i\,HML + \varepsilon_i$$

Con $s > 0$ el activo se comporta como empresa pequeña; con $h > 0$, como acción de valor, y con $h < 0$, de crecimiento. Los factores son portafolios largo-corto, así que sus primas se leen como retornos. Primas históricas en Estados Unidos (1963 a 2026, biblioteca de Kenneth French): mercado 7.2%, SMB 1.8% y HML 3.6% anual.

In [ ]:
# La minera es mediana y de valor: b = 1.1, s = 0.4, h = 0.5; primas de 5.5, 2.0 y 3.5
exigido_ff3 = retorno_multifactor(rf, [1.1, 0.4, 0.5], [5.5, 2.0, 3.5])
exigido_capm = capm(rf, 1.2, erp)
print(f"Exigido por CAPM = {exigido_capm:.2f}% -> alfa = {12.0 - exigido_capm:+.2f}")
print(f"Exigido por FF3  = {exigido_ff3:.2f}% -> alfa = {12.0 - exigido_ff3:+.2f}")

Salida esperada: 11.10% con alfa de $+0.90$ contra 13.10% con alfa de $-1.10$. Con el CAPM la minera parecía una oportunidad; con tres factores, su retorno extra era compensación por ser pequeña y de valor, y aún se queda corta. **El alfa es una propiedad del activo y del modelo.** Y para la unidad 1: con un $K_e$ de 13.1% en lugar de 11.1% el valor de la empresa cae; elegir el modelo de riesgo es una decisión de valoración.

Usos profesionales: atribución de retorno (exposición a factores contra selección), atribución de riesgo (de dónde viene el *tracking error*; semana 10), construcción de portafolios (inclinaciones a factores, ETF de *smart beta*), costo de capital y análisis de estilo.

## 8. Respuestas a los ítems de la clase

**1. B.** $\beta = \rho\,\sigma_i/\sigma_m = 0.6 \times 30/15 = 1.20$. La A multiplica por el cociente invertido; la C olvida la correlación.

**2. C.** Exigido: $4 + 1.4(9 - 4) = 11\%$. El pronóstico de 10% queda por debajo: alfa de $-1$, bajo la SML, sobrevalorada. Que rinda más que el mercado no basta: con beta de 1.4 tiene que rendir bastante más.

**3. B.** En un modelo macroeconómico los factores son sorpresas. La A describe un modelo fundamental como Fama-French; la C, uno estadístico.

**4. A.** $3 + 1.0(6) + 0.5(2) + (-0.2)(4) = 9.2\%$. La C se equivoca en el signo de la carga a HML; la B la ignora.

## 9. Lista de verificación

Sabes de esta semana si puedes: explicar por qué el tangente es el mercado y distinguir CML de SML; calcular un beta desde la correlación y un retorno exigido; leer un alfa como posición frente a la SML; descomponer la varianza en sistemática e idiosincrática y conectar el $R^2$; nombrar dos críticas al CAPM; enunciar los supuestos del APT y armar un arbitraje; distinguir las tres familias de modelos y qué es el intercepto en cada una; interpretar cargas a SMB y HML; y explicar por qué el alfa depende del modelo.